In [1]:
import pandas as pd
v = pd.read_csv("data/clean/stop_visits.csv",
                dtype={"route_id": str, "stop_id": str, "trip_id": str})
print(v.shape)
print(list(v.columns))
v.head()

(28708, 21)
['ts', 'trip_id', 'route_id', 'stop_id', 'stop_sequence', 'delay_sec', 'local_time', 'hour', 'weekday', 'outlier', 'route_long_name', 'stop_name', 'stop_lat', 'stop_lon', 'in_timetable', 'seq', 'date', 'trip_key', 'late', 'early', 'pos']


,ts,trip_id,route_id,stop_id,stop_sequence,delay_sec,local_time,hour,weekday,outlier,...,stop_name,stop_lat,stop_lon,in_timetable,seq,date,trip_key,late,early,pos
0,1792420579,33227025,518,3915,48,-198,2026-10-19 08:36:19-06:00,8,Monday,False,...,Mill Woods Transit Centre,53.457126,-113.428798,False,48,2026-10-19,2026-10-19_33227025,False,True,31+
1,1792420579,33224294,054,7902,63,-105,2026-10-19 08:36:19-06:00,8,Monday,False,...,West Clareview Transit Centre,53.602319,-113.411818,True,63,2026-10-19,2026-10-19_33224294,False,True,31+
2,1792420579,33225570,904,5205,48,-33,2026-10-19 08:36:19-06:00,8,Monday,False,...,Westmount Transit Centre,53.560984,-113.555641,False,48,2026-10-19,2026-10-19_33225570,False,False,31+
3,1792420579,33232891,719,4212,38,-89,2026-10-19 08:36:19-06:00,8,Monday,False,...,Century Park Transit Centre,53.457245,-113.515634,False,38,2026-10-19,2026-10-19_33232891,False,True,31+
4,1792420579,33234035,508,3704,28,-559,2026-10-19 08:36:19-06:00,8,Monday,False,...,Meadows Transit Centre,53.478246,-113.367476,False,28,2026-10-19,2026-10-19_33234035,False,True,11-30


#### Imagine 28,708 buses arriving at stops. Some arrived early, some on time, and some late. We want to answer one question: “How do the buses behave overall?” Looking at 28,708 numbers is impossible, so we squeeze them into a few numbers that anyone can understand.


In [2]:
s = v["delay_sec"]
print(s.describe(percentiles=[.1, .25, .5, .75, .9]).round(0))
print()
print("late (>5 min):   ", round((s > 300).mean() * 100, 1), "%")
print("early (>1 min):  ", round((s < -60).mean() * 100, 1), "%")
print("on time:         ", round(((s >= -60) & (s <= 300)).mean() * 100, 1), "%")
print("flagged outliers:", int(v["outlier"].sum()))

count    28708.0
mean       -81.0
std        226.0
min       -600.0
10%       -280.0
25%       -182.0
50%        -88.0
75%        -35.0
90%         98.0
max       3600.0
Name: delay_sec, dtype: float64

late (>5 min):    4.1 %
early (>1 min):   61.5 %
on time:          34.4 %
flagged outliers: 0


#### Numbers are good, but a picture shows the story in a second. This picture is called a histogram. It puts delays into small buckets, and each bar’s height shows how many stop visits fell into that bucket. Tall bars mean “lots of buses here.”


In [3]:
import plotly.express as px
fig = px.histogram(v[(s > -900) & (s < 1800)], x="delay_sec", nbins=80,
                   title="How early or late are buses at stops?")
fig.update_xaxes(title="Delay in seconds (negative = early, positive = late)")
fig.show()

##### so far we looked at all buses together. A bus company cares about which routes are good or bad. Think of a school with many classes. First we found the school’s average marks. Now we find each class’s marks, so we can see which class needs help. We also hide any route with fewer than 30 visits, because a result from 5 visits is just luck.

In [4]:
v["is_late"] = v["delay_sec"] > 300
v["is_early"] = v["delay_sec"] < -60
v["is_on_time"] = ~(v["is_late"] | v["is_early"])

routes = (v.groupby(["route_id", "route_long_name"])
            .agg(visits=("delay_sec", "size"),
                 median_delay_s=("delay_sec", "median"),
                 late_pct=("is_late", "mean"),
                 early_pct=("is_early", "mean"),
                 on_time_pct=("is_on_time", "mean"))
            .reset_index())

for c in ["late_pct", "early_pct", "on_time_pct"]:
    routes[c] = (routes[c] * 100).round(1)
routes["median_delay_s"] = routes["median_delay_s"].round(0)

routes = routes[routes["visits"] >= 30].sort_values("on_time_pct")
print(len(routes), "routes with at least 30 visits")
routes.head(10)

143 routes with at least 30 visits


,route_id,route_long_name,visits,median_delay_s,late_pct,early_pct,on_time_pct
49,401,Ordze - Downtown,131,-335.0,0.0,99.2,0.8
50,404,Ordze - University Of Alberta,53,-137.0,0.0,96.2,3.8
51,411,Bethel - Downtown,204,-201.0,0.0,96.1,3.9
87,524,Bonnie Doon - Holyrood,53,-206.0,0.0,94.3,5.7
86,523,Mill Woods - Downtown,237,-150.0,0.0,92.4,7.6
8,008,Abbottsfield - Downtown - University,423,-211.0,0.2,91.7,8.0
103,622,Stadium - Concordia University,46,-124.0,0.0,91.3,8.7
172,902,Nait - University,262,-175.0,0.0,90.5,9.5
215,F600,Fort Saskatchewan,73,-166.0,0.0,90.4,9.6
53,415,Bethel - Nait,78,-234.0,0.0,88.5,11.5


#### one list only tells half the story. We now look at the late end (which routes are most often late) and the best end (which routes are most on time). Think of a school again: after seeing the class with the lowest marks, you also check the top class and the class that is often late.

In [5]:
cols = ["route_id", "route_long_name", "visits", "median_delay_s",
        "late_pct", "early_pct", "on_time_pct"]

print("10 ROUTES MOST OFTEN LATE (more than 5 min)")
print(routes.sort_values(["late_pct", "median_delay_s"], ascending=False)[cols].head(10).to_string(index=False))

print()
print("10 ROUTES MOST ON TIME")
print(routes.sort_values("on_time_pct", ascending=False)[cols].head(10).to_string(index=False))

10 ROUTES MOST OFTEN LATE (more than 5 min)
route_id                   route_long_name  visits  median_delay_s  late_pct  early_pct  on_time_pct
     924        Jasper Place - Rio Terrace      49           224.0      38.8       20.4         40.8
     560  Spruce Grove - Downtown Edmonton     217            10.0      29.5       39.2         31.3
     713        Century Park - Twin Brooks     153           150.0      29.4       15.7         54.9
     052    Northgate - West Edmonton Mall     318           -29.0      28.9       43.7         27.4
    700X    Century Park - Heritage Valley     252           178.0      28.2        2.4         69.4
     915 West Edmonton Mall - Jasper Place     363            11.0      19.8       36.6         43.5
    110X           Eaux Claires - Downtown     280           -46.0      18.9       44.3         36.8
     201            Downtown Edmonton Loop     366           -28.0      16.4       41.3         42.3
     414    Bethel - University Of Alberta     

#### the “how sure are we?” range 
##### imagine tasting soup to check if it’s too salty. A tiny spoon from the pot might be unlucky and give you a wrong idea. A big spoon is more reliable. A confidence interval is the honest “guess range” around each route’s late percentage. The smaller the sample, the wider the range. Then we ask a fair question: which routes are late more often than the network average, even after allowing for luck? Those are the routes we can talk about with confidence.

In [6]:
from statsmodels.stats.proportion import proportion_confint

late_counts = v.groupby("route_id")["is_late"].sum()
routes["late_n"] = routes["route_id"].map(late_counts).astype(int)

lo, hi = proportion_confint(routes["late_n"], routes["visits"], method="wilson")
routes["late_lo"] = (lo * 100).round(1)
routes["late_hi"] = (hi * 100).round(1)

overall_late = (v["delay_sec"] > 300).mean() * 100
print("network average late rate:", round(overall_late, 1), "%")

reliably_late = routes[routes["late_lo"] > overall_late].sort_values("late_pct", ascending=False)
print(len(reliably_late), "routes are late more often than average, with confidence")
print()
print(reliably_late[["route_id", "route_long_name", "visits",
                     "late_pct", "late_lo", "late_hi"]].head(15).to_string(index=False))

network average late rate: 4.1 %
24 routes are late more often than average, with confidence

route_id                          route_long_name  visits  late_pct  late_lo  late_hi
     924               Jasper Place - Rio Terrace      49      38.8     26.4     52.8
     560         Spruce Grove - Downtown Edmonton     217      29.5     23.8     35.9
     713               Century Park - Twin Brooks     153      29.4     22.8     37.1
     052           Northgate - West Edmonton Mall     318      28.9     24.2     34.1
    700X           Century Park - Heritage Valley     252      28.2     23.0     34.0
     915        West Edmonton Mall - Jasper Place     363      19.8     16.1     24.2
    110X                  Eaux Claires - Downtown     280      18.9     14.8     23.9
     201                   Downtown Edmonton Loop     366      16.4     13.0     20.5
     414           Bethel - University Of Alberta     220      15.9     11.7     21.3
     926                 Lewis Farms - Stillwa

#### the “change the definition” test

##### Why: “late” means “more than 5 minutes” only because we chose that. A skeptic will ask: “If you’d said 3 minutes, or 7 minutes, would you get the same routes?” If yes, your finding is solid. If no, your finding depends on one arbitrary choice, and we must say so. Think of testing a bridge with a light truck and a heavy truck: you want it to hold both times.

In [7]:
thresholds = {"strict_3min": 180, "standard_5min": 300, "loose_7min": 420}

sens = pd.DataFrame({"route_id": routes["route_id"].values})
for name, t in thresholds.items():
    rate = v.assign(flag=v["delay_sec"] > t).groupby("route_id")["flag"].mean() * 100
    sens[name] = sens["route_id"].map(rate).round(1)

top = {name: set(sens.nlargest(15, name)["route_id"]) for name in thresholds}
common = top["strict_3min"] & top["standard_5min"] & top["loose_7min"]
print("routes in ALL three top-15 'most late' lists:", len(common), "of 15")
print(sorted(common))

print()
print("rank agreement (1.0 = exactly the same order):")
print(sens[list(thresholds)].corr(method="spearman").round(2))

routes in ALL three top-15 'most late' lists: 11 of 15
['002', '052', '110X', '120X', '201', '414', '560', '700X', '713', '915', '924']

rank agreement (1.0 = exactly the same order):
               strict_3min  standard_5min  loose_7min
strict_3min           1.00           0.94        0.92
standard_5min         0.94           1.00        0.99
loose_7min            0.92           0.99        1.00


#### the weekday vs weekend test

##### Why: this is your first hypothesis test, which asks: “Is a difference real, or could it just be luck?” Do buses behave differently on weekends? You now have one Saturday and one Sunday in the data, so we can check.
##### Think of flipping a coin. If you get 6 heads out of 10, you shrug. If you get 60 heads out of 100, you’d start to doubt the coin. The p-value is the answer to: “If weekdays and weekends were truly the same, how often would luck alone give a gap this big?”
##### Small p-value (below 0.05): luck alone rarely does this, so the difference is probably real.
##### Large p-value: it could easily be luck, so we can’t claim a difference.

In [8]:
from statsmodels.stats.proportion import proportions_ztest

v["is_weekend"] = v["weekday"].isin(["Saturday", "Sunday"])

g = v.groupby("is_weekend").agg(visits=("delay_sec", "size"),
                                late=("is_late", "sum"),
                                early=("is_early", "sum"),
                                median_delay_s=("delay_sec", "median"))
g["late_pct"] = (g["late"] / g["visits"] * 100).round(1)
g["early_pct"] = (g["early"] / g["visits"] * 100).round(1)
print(g)
print()

wk, we = g.loc[False], g.loc[True]
for label in ["late", "early"]:
    stat, p = proportions_ztest([we[label], wk[label]], [we["visits"], wk["visits"]])
    print(f"{label}: weekend {we[label + '_pct']}%  vs  weekday {wk[label + '_pct']}%   p-value = {p:.4f}")

            visits  late  early  median_delay_s  late_pct  early_pct
is_weekend                                                          
False        23345  1016  14297           -88.0       4.4       61.2
True          5363   170   3361           -87.0       3.2       62.7

late: weekend 3.2%  vs  weekday 4.4%   p-value = 0.0001
early: weekend 62.7%  vs  weekday 61.2%   p-value = 0.0526


#### delays through the day

##### Why: a bus company cares about when trouble happens. Imagine a school bell: the corridors are calm at 10 am but chaotic when the bell rings. We check whether buses are more often late or early at certain hours, like rush hour. We hide any hour with fewer than 100 visits, because a tiny sample would give a jumpy, unreliable line.

In [9]:
import plotly.express as px

by_hour = (v.groupby("hour")
             .agg(visits=("delay_sec", "size"),
                  late_pct=("is_late", "mean"),
                  early_pct=("is_early", "mean"),
                  median_delay_s=("delay_sec", "median"))
             .reset_index())
by_hour["late_pct"] = (by_hour["late_pct"] * 100).round(1)
by_hour["early_pct"] = (by_hour["early_pct"] * 100).round(1)
by_hour = by_hour[by_hour["visits"] >= 100]
print(by_hour.to_string(index=False))

fig = px.line(by_hour, x="hour", y=["late_pct", "early_pct"], markers=True,
              title="Share of stop visits that are late or early, by hour of day (Edmonton time)")
fig.update_xaxes(title="Hour of day (0-23)", dtick=2)
fig.update_yaxes(title="% of stop visits")
fig.show()

 hour  visits  late_pct  early_pct  median_delay_s
    0     193       1.0       69.4          -102.0
    5     477       1.0       63.3           -97.0
    6    1459       1.2       64.6           -91.0
    7    1894       2.6       61.8           -85.5
    8    1403       6.2       56.2           -70.0
    9    2055       3.0       66.1           -99.0
   10    2155       1.9       66.8          -109.0
   11    2079       2.6       65.8          -101.0
   12    1880       4.2       58.2           -84.0
   13    1781       4.0       61.8           -89.0
   14    1453       5.6       56.6           -73.0
   15    1891       7.0       52.9           -66.0
   16    1910       5.8       62.4           -88.0
   17    1728       7.2       59.0           -80.5
   18    1464       4.7       57.4           -75.0
   19     917       8.6       56.3           -77.0
   20    1285       5.1       59.6           -85.0
   21    1347       2.8       67.9          -112.0
   22     665       2.1       6